# OmicsFM API — inference example
Load a trained checkpoint and run inference on the flashlfq proteomics train set.

Two entry points:
- `compute_sst(...)` → one whole-sample (SST) embedding per sample.
- `predict(...)` → predict held-out expression bins + metrics (the masked-modeling eval).

In [2]:
%load_ext autoreload
%autoreload 2

import os
from omicsfm.hub import get_checkpoint, get_dataset, omicsfm_home

# The checkpoint config stores its FASTA and ESM-C cache paths relative to the
# repository root, so resolve them from there rather than the notebook folder.
os.chdir(omicsfm_home())
print("cwd:", os.getcwd())

from omicsfm.api import compute_sst, predict, visualize_sst
from omicsfm.data import ExpressionDataset

# Downloaded from the Hub on first use, then reused from disk.
CKPT = str(get_checkpoint("proteomics_esmc"))            # 111 MB
DATA = str(get_dataset("proteomics_uniprot", "train"))   # 652 MB
print("checkpoint:", CKPT)
print("data      :", DATA)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
cwd: c:\Users\sander\OneDrive\Bureaublad\Projects\prot_GPT\code\protgpt


## 1. `compute_sst` — whole-sample (SST) embeddings
Every detected protein is used as context, so each SST embedding summarizes the full proteome. Reads num_bins/detect_groups/ESM-C settings from the checkpoint config and rebuilds the ESM-C lookup for this dataset's proteins.

In [3]:
out = compute_sst(CKPT, DATA, device="cuda", batch_size=64, num_workers=0)
print("SST embeddings:", out["sst_emb"].shape)   # (n_samples, d_model)
print("n samples     :", len(out["sample_ids"]))
print("sample id [0]  :", out["sample_ids"][0])

Running model: 100%|██████████| 707/707 [00:39<00:00, 17.86it/s]


SST embeddings: (45188, 256)
n samples     : 45188
sample id [0]  : PXD000004_HSA


## 2. `predict` — predicted bins + metrics
Holds out a fraction of each sample's features (context/target split from the training config) and predicts their bins with both heads. Returns per-sample predictions + aggregate MSE/MAE.

In [4]:
ev = predict(CKPT, DATA, device="cuda", batch_size=64, num_workers=0)
print(ev["metrics"])             # ctx_mse, sst_mse, ctx_mae, sst_mae, n_predictions
print("pred_sst[0][:5] :", ev["pred_sst"][0][:5])
print("true_bins[0][:5]:", ev["true_bins"][0][:5])

KeyboardInterrupt: 

## 3. Metadata for coloring
`out["sample_ids"]` lines up row-for-row with `out["sst_emb"]` and with the dataset's `obs` metadata (`sample_id` = `<pxd>_<run>`).

In [5]:
ds = ExpressionDataset(DATA, num_bins=10, detect_groups=True, max_group_size=1)
print("obs columns:", list(ds.obs.columns))
ds.obs.head(3)

obs columns: ['pxd', 'run', 'organism', 'organism_evidence', 'tissue', 'tissue_evidence', 'disease', 'disease_evidence', 'cell_part', 'cell_part_evidence', 'cell_line', 'cell_line_evidence', 'instrument', 'instrument_evidence', 'fragmentation', 'fragmentation_evidence', 'enzymes', 'enzymes_evidence', 'modifications', 'modifications_evidence', 'collision_energy', 'collision_energy_evidence', 'gradient_time_min', 'gradient_time_min_evidence', 'lc_column', 'lc_column_evidence', 'acquisition', 'acquisition_evidence', 'labeling', 'labeling_evidence', 'ionization', 'ionization_evidence', 'treatment_type', 'treatment_type_evidence', 'treatment_name', 'treatment_name_evidence', 'treatment_class', 'treatment_class_evidence', 'fractionation', 'fractionation_evidence', 'enrichment', 'enrichment_evidence', 'split']


,pxd,run,organism,organism_evidence,tissue,tissue_evidence,disease,disease_evidence,cell_part,cell_part_evidence,...,treatment_type_evidence,treatment_name,treatment_name_evidence,treatment_class,treatment_class_evidence,fractionation,fractionation_evidence,enrichment,enrichment_evidence,split
sample_id,,,,,,,,,,,,,,,,,,,,,
PXD000004_HSA,PXD000004,HSA,Homo sapiens,agent,brain,agent; MLM,healthy,agent,nan,nan,...,NaN,NaN,NaN,nan,nan,gel electrophoresis; True,agent,biochemical fractionation; synaptic microdomains,agent,train
PXD000004_HSA2,PXD000004,HSA2,Homo sapiens,agent,brain,agent; MLM,healthy,agent,nan,nan,...,NaN,NaN,NaN,nan,nan,gel electrophoresis; True,agent,biochemical fractionation; synaptic microdomains,agent,train
PXD000004_HSA3,PXD000004,HSA3,Homo sapiens,agent,brain,agent; MLM,healthy,agent,nan,nan,...,NaN,NaN,NaN,nan,nan,gel electrophoresis; True,agent,biochemical fractionation; synaptic microdomains,agent,train


In [27]:
TISSUE_MAP = {
    'blood':           ['blood', 'blood serum', 'blood plasma'],
    'platelets':       ['platelets'],
    'monocytes':       ['T cells', 'B cells', 'NK cells', 'macrophage', 'monocytes', 'CD4+ T cells', 'CD8+ T cells'],
    'lymphoid':        ['spleen', 'lymph node', 'bone marrow', 'tonsil'],
    'brain':           ['brain', 'frontal cortex', 'cerebellum', 'spinal cord', 'occipital lobe',
                        'temporal lobe', 'parietal lobe', 'substantia nigra', 'retina'],
    'liver':           ['liver'],
    'kidney':          ['kidney', 'cortex of kidney'],
    'lung':            ['lung'],
    'heart':           ['heart'],
    'thyroid':         ['thyroid gland'],
    'pancreas':        ['pancreas'],
    'adrenal':         ['adrenal gland'],
    'GI tract':        ['colon', 'small intestine', 'stomach', 'esophagus', 'rectum', 'gut',
                        'duodenum', 'gallbladder', 'vermiform appendix'],
    'testis':          ['testis', 'sperm', 'seminal plasma'],
    'ovary':           ['ovary'],
    'breast':          ['breast'],
    'skin':            ['skin'],
    'skeletal muscle': ['skeletal muscle'],
    'cartilage':       ['cartilage'],
    'stem cells':      ['stem cells'],
    'urine':           ['urine'],
    'saliva':          ['saliva', 'salivary gland'],
    'body fluids':     ['sweat', 'tear fluid', 'synovial fluid', 'milk', 'sputum',
                        'bronchoalveolar lavage fluid', 'peritoneal dialysis fluid', 'pleural fluid',
                        'cervicovaginal fluid', 'dental plaque', 'feces', 'follicular fluid'],
}

In [28]:
# reverse: fine tissue -> coarse class
fine_to_class = {fine: cls for cls, fines in TISSUE_MAP.items() for fine in fines}

obs = ds.obs.copy()
obs["tissue_class"] = obs["tissue"].map(fine_to_class)

mapped = obs["tissue_class"].notna()
print(f"mapped {mapped.sum():,}/{len(obs):,} samples into {obs['tissue_class'].nunique()} classes")
print("unmapped tissues:", sorted(obs.loc[~mapped, 'tissue'].dropna().unique()))

mapped 16,347/45,188 samples into 23 classes
unmapped tissues: ['B cells; T cells', 'B cells; blood', 'B cells; blood; epithelial cell; melanocyte', 'CD8+ T cells; PBMCs', 'PBMCs', 'PBMCs; stem cells', 'adipose tissue', 'adipose tissue; blood; bone; skeletal muscle', 'adipose tissue; liver', 'anus', 'blood plasma; blood serum', 'blood plasma; brain', 'blood plasma; cerebrospinal fluid', 'blood plasma; duodenum', 'blood plasma; follicular fluid', 'blood plasma; intervertebral disk', 'blood plasma; liver', 'blood; NK cells', 'blood; PBMCs', 'blood; T cells', 'blood; blood serum', 'blood; bone marrow', 'blood; erythrocyte', 'blood; fibroblast; kidney', 'blood; fibroblast; skeletal muscle', 'blood; liver', 'blood; macrophage', 'blood; monocytes', 'blood; neutrophil', 'blood; platelets', 'blood; saliva; sperm', 'bone marrow; brain; cerebrospinal fluid; cervicovaginal fluid; colon; heart; kidney; liver; lung; lymph node; placenta; seminal plasma; skin; spleen; sweat; synovial fluid; testis; 

In [56]:
import numpy as np

# Focus on well-defined, distinct tissue classes for a legible UMAP — drop the heterogeneous
# grab-bag ('body fluids') and 'stem cells' / 'cartilage'. Edit CURATED and re-run to change the set.
CURATED = [
    "blood", "platelets", "lymphoid", "monocytes",            # blood / immune
    "brain", "heart", "lung", "liver", "kidney", "pancreas",  # solid organs
    "thyroid", "adrenal", "GI tract", "skin", "skeletal muscle",
    "breast", "ovary", "testis",
    "saliva", "urine",                                        # body fluids
]
tc = obs["tissue"].map(fine_to_class)            # recompute from the full map (idempotent)
obs["tissue_class"] = tc.where(tc.isin(CURATED))
mapped = obs["tissue_class"].notna()

keep = set(obs.index[mapped])
m = np.array([sid in keep for sid in out["sample_ids"]])
out_mapped = {"sst_emb": out["sst_emb"][m],
              "sample_ids": [s for s, k in zip(out["sample_ids"], m) if k]}
print(f"plotting {m.sum():,} of {len(m):,} samples across "
      f"{obs.loc[mapped, 'tissue_class'].nunique()} curated tissue classes")

plotting 15,694 of 45,188 samples across 20 curated tissue classes


## 4. Visualize the SST embeddings (interactive plotly)
We focus on the curated set of well-defined, distinct tissue classes (edit `CURATED` above) so
the clusters are legible — the heterogeneous grab-bag (`body fluids`) and ambiguous classes are
dropped. `visualize_sst` reduces with UMAP/PCA/t-SNE and returns an interactive plotly figure.
`color_by` colours the points; `hover` adds extra metadata to the tooltip. UMAP defaults:
`n_neighbors=75, min_dist=0.3, random_state=42` (override via kwargs).

In [57]:
# Explicit colour per class, in legend order — TUNE THESE. Edit a hex to make two neighbours
# more distinguishable, or reorder rows to change the legend/gradient sequence. blood = red.
CLASS_COLORS = {
    "blood":           "#e6194b",   # red
    "lymphoid":        "#da6c7f",   # yellow
    "platelets":       "#f58231",   # orange
    "monocytes":       "#ffb000",   # amber
    "urine":           "#cdbf6a",   # khaki (fluid)
    "skeletal muscle": "#808000",   # olive
    "heart":           "#bfef45",   # green
    "brain":           "#3cb44b",   # yellow-green
    "lung":            "#1f9e89",   # teal-green
    "liver":           "#30505F",   # teal
    "pancreas":        "#42d4f4",   # cyan
    "saliva":          "#9fb8c4",   # pale blue-grey (fluid)
    "GI tract":        "#4363d8",   # blue
    "kidney":          "#3b1fb0",   # indigo
    "adrenal":         "#911eb4",   # violet
    "thyroid":         "#c354c3",   # purple
    "skin":            "#f032e6",   # magenta
    "ovary":           "#ff6fb5",   # pink
    "testis":          "#9a6324",   # brown
    "breast":          "#dfb58c",   # apricot
}
order = [c for c in CLASS_COLORS if c in set(obs.loc[mapped, "tissue_class"])]
palette = [CLASS_COLORS[c] for c in order]

fig = visualize_sst(
    out_mapped,
    metadata=obs,                 # carries the new 'tissue_class' column
    color_by="tissue_class",
    color_order=order,
    palette=palette,
    hover=["tissue", "acquisition", "split"],
    method="umap",
    width=1600, height=1200,
    n_neighbors=150, min_dist=0.7, random_state=0, metric="euclidean",
)

c:\Users\sander\miniconda3\envs\prot_gpt\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 5. Attention maps

The transformer layers are the only place where information moves *between* proteins/genes, so the attention weights are a direct read-out of which proteins/genes the model uses to predict which others. Accumulated over many samples, the attention between two proteins becomes a score for how related they are — a candidate ranking for protein–protein interactions (PPIs).

But attention heads are **specialized**: each layer has multiple heads, and they do not all encode biology. Some heads carry a strong, specific PPI signal; many others are essentially noise — and which is which isn't known in advance. So the workflow is two-stage:

1. **Evaluate every head** against known PPIs to find where the signal lives.
2. **Accumulate the full map** for the winning head only.

`eval_attention_heads` benchmarks **every (layer, head)** against curated PPI databases (CORUM, STRING, KEGG, Reactome, GO-CC), scoring each by enrichment AUC — how strongly its top-attended pairs are enriched for *known* interactions. With `db="combined"` it scores all five and averages them, so we pick the head that generalizes across databases rather than overfitting one.

Note: this makes one pass per layer (reading attention disables the fused-attention kernel),
so it's the slow step — run it once and read off the best head from the heatmap.

#### Find which heads hold the singal

In [ ]:
from omicsfm.api import eval_attention_heads
res = eval_attention_heads(CKPT, DATA, db="combined")

In [ ]:
res['best']

#### Accumulate the full map of the best head
Now that we know which head carries the signal, we run inference **once more, accumulating only that head's** protein×protein attention over the full proteome (one data pass). The result is a symmetric `P×P` score matrix: for any protein, its highest-scoring partners are the model's top PPI candidates.

In [ ]:
from omicsfm.api import attention_map

best = (1, 4)
m = attention_map(CKPT, DATA, head=best, batch_size=32, num_workers=0, symmetrize=True)

In [ ]:
from omicsfm.api import enrichment_curves

curves = enrichment_curves(m, db="combined")